# 第12周：主线结果整合、主图草稿与阶段论文成形

**本周目标**（任务书）：在不依赖 scVI/UCE 的情况下，形成一套可支撑毕业论文答辩的完整结果。

本周交付物：

| 交付物 | 路径 |
|---|---|
| 主图草稿 Fig1–Fig4 | `figures/main/Fig1_draft.pdf` ~ `Fig4_draft.pdf` |
| 样本汇总表 | `tables/sample_summary.xlsx` |
| 结论审计表 | `docs/claim_audit_v1.xlsx` |
| 方法与结果草稿 | `manuscript/methods_results_v1.docx` |
| 最小流程复跑核验 | `results/reproducibility/minimal_rerun_check.csv` |

执行要求对照：
- 整合数据流程、QC、PCA、pseudobulk、差异表达、富集和 PCA 衰老相似性 → Cell 2–8
- 至少 4 张主图草稿 → Cell 5–8（Fig1 研究设计与QC、Fig2 样本级PCA年龄趋势、Fig3 差异表达、Fig4 通路模块）
- 样本汇总表 → Cell 4；claim audit → Cell 9；方法/结果草稿 → Cell 10
- 独立复跑从冻结数据到主要结果的最小流程 → Cell 11
- 10分钟阶段汇报提纲与"每个 n 代表什么"清单 → Cell 12（Markdown）

统计单位声明：除 QC 漏斗以细胞为单位外，**所有下游分析的 n 均为独立小鼠数**（mouse-level）。


In [1]:
# Cell 1 ── 环境、路径与全局设置
import warnings
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from scipy import stats
from pathlib import Path
warnings.filterwarnings('ignore')

BASE = Path('..')
RES = BASE / 'results'
FIG_MAIN = BASE / 'figures' / 'main'
TABLES = BASE / 'tables'
DOCS = BASE / 'docs'
MANU = BASE / 'manuscript'
for d in (FIG_MAIN, TABLES, DOCS, MANU, RES / 'reproducibility'):
    d.mkdir(parents=True, exist_ok=True)

SEED = 42
AGE_MAP = {1: 'young', 3: 'young', 18: 'middle', 21: 'middle', 24: 'old', 30: 'old'}
AGE_ORDER = ['young', 'middle', 'old']
AGE_COLORS = {'young': '#4C9F70', 'middle': '#E8A838', 'old': '#C0504D'}
N_PC_AXIS = 30            # 衰老方向使用的 PC 数（与第10/11周一致）
plt.rcParams.update({'figure.dpi': 150, 'axes.spines.top': False,
                     'axes.spines.right': False, 'font.size': 9})
print('输出目录已就绪')


输出目录已就绪


In [2]:
# Cell 2 ── 加载全部主线结果（冻结数据 → 各周产出）
pca_df = pd.read_csv(RES / 'pca' / 'mouse_level_pca.csv', index_col=0).reset_index()
pca_df['node'] = pca_df['tissue'] + '__' + pca_df['major_cell_type']

scores = pd.read_csv(RES / 'aging_axis' / 'pca_scores.csv')
cosine = pd.read_csv(RES / 'aging_axis' / 'cosine_similarity.csv', index_col=0)
NODES = list(cosine.columns)

de_summary = pd.read_csv(RES / 'de' / 'DE_summary_old_vs_young.tsv', sep='\t')
nes = pd.read_csv(RES / 'enrichment' / 'pathway_nes_matrix.tsv', sep='\t', index_col=0)
fdr_mat = pd.read_csv(RES / 'enrichment' / 'pathway_fdr_matrix.tsv', sep='\t', index_col=0)
modules = pd.read_csv(RES / 'enrichment' / 'module_classification.tsv', sep='\t')
sens = pd.read_csv(RES / 'sensitivity' / 'sensitivity_summary.tsv', sep='\t')
boot = pd.read_csv(RES / 'sensitivity' / 'bootstrap_ci.csv')
lib = pd.read_csv(RES / 'pseudobulk' / 'library_qc.csv')
excl = pd.read_csv(RES / 'de' / 'sample_exclusion_audit.tsv', sep='\t')
filt = pd.read_csv(RES / 'qc' / 'filtering_audit.csv')
age_change = pd.read_csv(RES / 'qc' / 'age_cell_change.csv')
cohort = pd.read_csv(RES / 'qc' / 'final_cohort_matrix.csv')

print('分析节点（7个）:', NODES)
print('冻结队列细胞数:', cohort['cell_number'].sum())
print('mouse-level PCA 行数:', len(pca_df), '| 有效PC行数:', pca_df.dropna(subset=['PC1']).shape[0])


分析节点（7个）: ['Heart_and_Aorta__endothelial', 'Heart_and_Aorta__fibroblast/stromal', 'Kidney__endothelial', 'Kidney__fibroblast/stromal', 'Liver__endothelial', 'Lung__endothelial', 'Lung__fibroblast/stromal']
冻结队列细胞数: 106368
mouse-level PCA 行数: 1260 | 有效PC行数: 110


In [3]:
# Cell 3 ── 加载各节点差异表达结果，统计共享/特异基因
def de_filename(node):
    t, c = node.split('__')
    return '%s_%s_old_vs_young_DE.tsv' % (t, c.replace('/', '_'))

def as_bool(s):
    if s.dtype == bool:
        return s
    return s.astype(str).str.strip().str.lower().map({'true': True, 'false': False}).fillna(False)

de_tables, sig_genes = {}, {}
for nd in NODES:
    p = RES / 'de' / de_filename(nd)
    if not p.exists():
        print('缺失:', p)
        continue
    d = pd.read_csv(p, sep='\t')
    d = d[as_bool(d['significant'])].copy()
    de_tables[nd] = d
    sig_genes[nd] = dict(zip(d['gene'], d['direction']))

gene_by_node = {}
for nd, m in sig_genes.items():
    for g, direction in m.items():
        gene_by_node.setdefault(g, {})[nd] = direction
shared_genes = {g: v for g, v in gene_by_node.items() if len(v) >= 2}
print('各节点显著DE基因数:', {nd: len(m) for nd, m in sig_genes.items()})
print('在>=2个节点显著的共享基因数:', len(shared_genes))


各节点显著DE基因数: {'Heart_and_Aorta__endothelial': 0, 'Heart_and_Aorta__fibroblast/stromal': 0, 'Kidney__endothelial': 2, 'Kidney__fibroblast/stromal': 56, 'Liver__endothelial': 1, 'Lung__endothelial': 0, 'Lung__fibroblast/stromal': 9}
在>=2个节点显著的共享基因数: 0


In [4]:
# Cell 4 ── 样本汇总表 tables/sample_summary.xlsx（n = 独立小鼠）
try:
    import openpyxl  # noqa
except ImportError:
    import sys, subprocess
    subprocess.run([sys.executable, '-m', 'pip', 'install', 'openpyxl', '-q'], check=True)

lib2 = lib.copy()
lib2['node'] = lib2['tissue'] + '__' + lib2['major_cell_type']
lib2['age_group'] = lib2['age_months'].map(AGE_MAP)
excl_ids = set(excl['sample_id'])
lib2['de_excluded'] = lib2['sample_id'].isin(excl_ids)
de_status_map = {(r.tissue, r.cell_type): r.status for r in de_summary.itertuples()}
lib2['de_node_status'] = [de_status_map.get((t, c), 'NA')
                          for t, c in zip(lib2['tissue'], lib2['major_cell_type'])]

# —— Sheet 1: 逐样本（mouse × tissue × cell_type）
if 'sex' in lib2.columns:
    sheet_samples = lib2[['sample_id', 'mouse.id', 'tissue', 'major_cell_type',
                          'age_months', 'age_group', 'sex', 'n_cells', 'library_size',
                          'mean_sample_correlation', 'sample_outlier', 'outlier_reason',
                          'de_excluded', 'de_node_status']]
else:
    sheet_samples = lib2[['sample_id', 'mouse.id', 'tissue', 'major_cell_type',
                          'age_months', 'age_group', 'n_cells', 'library_size',
                          'mean_sample_correlation', 'sample_outlier', 'outlier_reason',
                          'de_excluded', 'de_node_status']]

# —— Sheet 2: 逐节点汇总
node_rows = []
for nd in NODES:
    t, c = nd.split('__')
    sub_lib = lib2[(lib2['node'] == nd) & (lib2['n_cells'] > 0) & (~lib2['de_excluded'])]
    ds = de_summary[(de_summary['tissue'] == t) & (de_summary['cell_type'] == c)]
    sc = scores[scores['node'] == nd]
    row = {
        'node': nd,
        'n_mice_total': sub_lib['mouse.id'].nunique(),
        'n_mice_young': sub_lib[sub_lib['age_group'] == 'young']['mouse.id'].nunique(),
        'n_mice_middle': sub_lib[sub_lib['age_group'] == 'middle']['mouse.id'].nunique(),
        'n_mice_old': sub_lib[sub_lib['age_group'] == 'old']['mouse.id'].nunique(),
        'n_cells_total': int(sub_lib['n_cells'].sum()),
        'de_status': ds['status'].iloc[0] if len(ds) else 'NA',
        'de_n_samples': ds['n_samples'].iloc[0] if len(ds) else np.nan,
        'de_significant_genes': ds['significant_genes'].iloc[0] if len(ds) else np.nan,
        'de_up_genes': ds['up_genes'].iloc[0] if len(ds) else np.nan,
        'de_down_genes': ds['down_genes'].iloc[0] if len(ds) else np.nan,
        'aging_score_spearman_rho': sc['spearman_rho'].iloc[0] if len(sc) else np.nan,
        'aging_score_spearman_p': sc['spearman_p'].iloc[0] if len(sc) else np.nan,
    }
    node_rows.append(row)
sheet_nodes = pd.DataFrame(node_rows)

# —— Sheet 3/4: QC 漏斗与按年龄保留
sheet_qc = filt.copy()
sheet_qc_age = age_change.copy()

out_xlsx = TABLES / 'sample_summary.xlsx'
with pd.ExcelWriter(out_xlsx, engine='openpyxl') as w:
    sheet_samples.to_excel(w, sheet_name='per_sample', index=False)
    sheet_nodes.to_excel(w, sheet_name='per_node', index=False)
    sheet_qc.to_excel(w, sheet_name='qc_funnel', index=False)
    sheet_qc_age.to_excel(w, sheet_name='qc_by_age', index=False)
print('已保存:', out_xlsx)
print(sheet_nodes[['node', 'n_mice_total', 'n_mice_young', 'n_mice_old',
                   'de_status', 'de_significant_genes', 'aging_score_spearman_rho']].to_string(index=False))


已保存: ..\tables\sample_summary.xlsx
                               node  n_mice_total  n_mice_young  n_mice_old de_status  de_significant_genes  aging_score_spearman_rho
       Heart_and_Aorta__endothelial            11             2           4        OK                   0.0                  0.854615
Heart_and_Aorta__fibroblast/stromal            11             2           4        OK                   9.0                  0.831517
                Kidney__endothelial            14             5           4        OK                  55.0                  0.874378
         Kidney__fibroblast/stromal            14             5           4        OK                 266.0                  0.908094
                 Liver__endothelial            11             5           4        OK                  12.0                  0.804398
                  Lung__endothelial            14             4           4        OK                  25.0                  0.906731
           Lung__fibroblast

In [5]:
# Cell 5 ── Figure 1 草稿：研究设计、QC流程与样本覆盖
fig = plt.figure(figsize=(13, 8))
gs = gridspec.GridSpec(2, 3, figure=fig, hspace=0.42, wspace=0.35)

# (a) 分析层级示意（文字面板）
ax = fig.add_subplot(gs[0, 0]); ax.axis('off')
design_text = (
    'Study design\n\n'
    'Data: Tabula Muris Senis (FACS)\n'
    'Frozen cohort: %d cells, %d genes (HVG)\n\n'
    'Tissues (5): Heart_and_Aorta, Kidney,\nLiver, Lung, (Fat: no young mice)\n'
    'Cell types (2): endothelial,\nfibroblast/stromal\n\n'
    'Analysis nodes: 7 (tissue x cell type)\n'
    'Statistical unit: independent mice\n\n'
    'Pipeline: QC -> sample-level PCA ->\npseudobulk + edgeR -> fgsea ->\n'
    'aging axis -> sensitivity' % (cohort['cell_number'].sum(), 3000))
ax.text(0.02, 0.98, design_text, va='top', ha='left', fontsize=8.5, family='monospace')
ax.set_title('(a) Study design', loc='left', fontweight='bold')

# (b) QC 漏斗（单位：细胞）
ax = fig.add_subplot(gs[0, 1])
steps = filt['step'].tolist()
counts = filt['cells_after'].tolist()
labels = {'raw': 'Raw', 'n_genes_200_6000': 'n_genes\n200-6000',
          'median_3MAD': 'median\n3-MAD', 'mitochondrial_filter': 'mito\nfilter'}
ax.bar(range(len(steps)), counts, color='#7FA8C9')
for i, (s, v) in enumerate(zip(steps, counts)):
    ax.text(i, v + 1500, '%d' % v, ha='center', fontsize=8)
ax.set_xticks(range(len(steps)))
ax.set_xticklabels([labels.get(s, s) for s in steps], fontsize=8)
ax.set_ylabel('Cells retained')
ax.set_title('(b) QC funnel (unit: cells)', loc='left', fontweight='bold')

# (c) 按年龄的细胞保留
ax = fig.add_subplot(gs[0, 2])
x = np.arange(len(age_change))
ax.bar(x - 0.2, age_change['before'], width=0.4, color='#BBBBBB', label='before QC')
ax.bar(x + 0.2, age_change['after'], width=0.4, color='#7FA8C9', label='after QC')
ax.set_xticks(x)
ax.set_xticklabels(age_change['age'].astype(str) + 'm', fontsize=8)
ax.set_ylabel('Cells')
ax.legend(fontsize=8)
ax.set_title('(c) Cells retained by age', loc='left', fontweight='bold')

# (d) 各节点小鼠覆盖（n = 独立小鼠，按年龄组）
ax = fig.add_subplot(gs[1, :2])
pca_ok = pca_df[pca_df['node'].isin(NODES)].dropna(subset=['PC1'])
cov = (pca_ok.groupby(['node', 'age_group'])['mouse.id'].nunique()
       .unstack().reindex(columns=AGE_ORDER).fillna(0).loc[NODES])
bottom = np.zeros(len(NODES))
for ag in AGE_ORDER:
    ax.barh(range(len(NODES)), cov[ag], left=bottom, color=AGE_COLORS[ag], label=ag)
    bottom += cov[ag].values
for i, nd in enumerate(NODES):
    ax.text(bottom[i] + 0.15, i, '%d mice' % int(bottom[i]), va='center', fontsize=8)
ax.set_yticks(range(len(NODES)))
ax.set_yticklabels(NODES, fontsize=8)
ax.set_xlabel('Number of mice (with valid sample-level PCA)')
ax.legend(fontsize=8, ncol=3)
ax.set_title('(d) Mouse coverage per analysis node (n = mice)', loc='left', fontweight='bold')

# (e) 排除样本记录
ax = fig.add_subplot(gs[1, 2]); ax.axis('off')
excl_text = 'Excluded samples (Week 7 outlier QC):\n\n'
for r in excl.itertuples():
    excl_text += '- %s\n  (%s)\n' % (r.sample_id, r.age_group)
ax.text(0.02, 0.98, excl_text, va='top', ha='left', fontsize=7.5, family='monospace')
ax.set_title('(e) Sample exclusions', loc='left', fontweight='bold')

fig.suptitle('Figure 1 (draft): study design, QC pipeline and sample coverage', y=0.995, fontsize=11)
fig.savefig(FIG_MAIN / 'Fig1_draft.pdf', bbox_inches='tight')
fig.savefig(FIG_MAIN / 'Fig1_draft.png', bbox_inches='tight')
plt.close(fig)
print('已保存 figures/main/Fig1_draft.pdf / .png')


已保存 figures/main/Fig1_draft.pdf / .png


In [6]:
# Cell 6 ── Figure 2 草稿：样本级PCA衰老趋势 + 方向相似度热图
fig = plt.figure(figsize=(14, 7.5))
gs = gridspec.GridSpec(2, 8, figure=fig, hspace=0.55, wspace=0.9,
                       width_ratios=[1, 1, 1, 1, 1, 1, 1, 1.3])

# (a) 每个节点：衰老得分 vs 月龄（n = 独立小鼠）
for i, nd in enumerate(NODES):
    ax = fig.add_subplot(gs[i // 4, i % 4])
    sub = scores[scores['node'] == nd]
    for ag, g in sub.groupby('age_group'):
        ax.scatter(g['age_months'], g['aging_score'], s=28,
                   color=AGE_COLORS.get(ag, '#999999'), label=ag, edgecolors='k', linewidths=0.3)
    rho, p = sub['spearman_rho'].iloc[0], sub['spearman_p'].iloc[0]
    ax.set_title('%s\nrho=%.2f, p=%.1g, n=%d' % (nd.replace('__', ' / '), rho, p, len(sub)),
                 fontsize=7.5)
    ax.set_xlabel('Age (months)', fontsize=7)
    if i % 4 == 0:
        ax.set_ylabel('Aging score', fontsize=7)
    ax.tick_params(labelsize=6.5)

# (b) 衰老方向余弦相似度热图
ax = fig.add_subplot(gs[:, 4:8])
short = [nd.replace('__', '\n') for nd in NODES]
sns.heatmap(cosine.values, ax=ax, cmap='RdBu_r', center=0, vmin=-1, vmax=1,
            xticklabels=short, yticklabels=short, annot=True, fmt='.2f',
            annot_kws={'size': 7}, cbar_kws={'label': 'cosine similarity'})
ax.set_xticklabels(ax.get_xticklabels(), fontsize=7, rotation=45, ha='right')
ax.set_yticklabels(ax.get_yticklabels(), fontsize=7, rotation=0)
ax.set_title('(b) Cosine similarity of aging directions\n(d = mean_old - mean_young, PC1-%d, n = mice per group)' % N_PC_AXIS,
             fontsize=8.5, loc='left')

fig.suptitle('Figure 2 (draft): sample-level PCA aging trend and direction similarity', y=0.995, fontsize=11)
fig.savefig(FIG_MAIN / 'Fig2_draft.pdf', bbox_inches='tight')
fig.savefig(FIG_MAIN / 'Fig2_draft.png', bbox_inches='tight')
plt.close(fig)
print('已保存 figures/main/Fig2_draft.pdf / .png')


已保存 figures/main/Fig2_draft.pdf / .png


In [7]:
# Cell 7 ── Figure 3 草稿：pseudobulk差异表达与共享/特异基因
from matplotlib.colors import ListedColormap

fig = plt.figure(figsize=(13, 8))
gs = gridspec.GridSpec(2, 2, figure=fig, hspace=0.4, wspace=0.3,
                       height_ratios=[1, 1.25])

# (a) 各节点 DE 基因数（n = 独立小鼠，见每节点样本数）
ax = fig.add_subplot(gs[0, 0])
ok = de_summary[de_summary['status'] == 'OK'].copy()
ok['node'] = ok['tissue'] + '__' + ok['cell_type']
ok = ok.set_index('node').loc[[n for n in NODES if n in set(ok['node'])]]
x = np.arange(len(ok))
ax.bar(x - 0.2, ok['up_genes'], width=0.4, color='#C0504D', label='Up in old')
ax.bar(x + 0.2, ok['down_genes'], width=0.4, color='#4C72B0', label='Down in old')
for i, r in enumerate(ok.itertuples()):
    ax.text(i, max(r.up_genes, r.down_genes) + 5, 'n=%s' % r.n_samples, ha='center', fontsize=7)
ax.set_xticks(x)
ax.set_xticklabels([n.replace('__', '\n') for n in ok.index], fontsize=7, rotation=0)
ax.set_ylabel('Significant genes (FDR<0.05)')
ax.legend(fontsize=8)
ax.set_title('(a) DE gene counts per node (edgeR ~ age_group + sex)', loc='left', fontweight='bold')

# (b) 共享程度分布：在 k 个节点显著的基因数
ax = fig.add_subplot(gs[0, 1])
k_counts = pd.Series([len(v) for v in gene_by_node.values()]).value_counts().sort_index()
ax.bar(k_counts.index, k_counts.values, color='#7FA8C9')
for k, v in k_counts.items():
    ax.text(k, v + 3, str(v), ha='center', fontsize=8)
ax.set_xlabel('Number of nodes where gene is significant')
ax.set_ylabel('Number of genes')
ax.set_title('(b) Cross-node sharing of DE genes', loc='left', fontweight='bold')

# (c) 共享基因方向一致性热图（取出现在>=2节点的基因，最多前18个）
ax = fig.add_subplot(gs[1, :])
shared_sorted = sorted(shared_genes.items(), key=lambda kv: -len(kv[1]))[:18]
glist = [g for g, _ in shared_sorted]
mat = np.full((len(glist), len(NODES)), np.nan)
dir_code = {'Up': 1, 'Down': -1}
for i, (g, v) in enumerate(shared_sorted):
    for j, nd in enumerate(NODES):
        if nd in v:
            mat[i, j] = dir_code.get(v[nd], 0)
cmap3 = ListedColormap(['#4C72B0', '#DDDDDD', '#C0504D'])
sns.heatmap(pd.DataFrame(mat, index=glist, columns=[n.replace('__', '\n') for n in NODES]),
            ax=ax, cmap=cmap3, vmin=-1, vmax=1, cbar=False, linewidths=0.6, linecolor='white')
for i in range(mat.shape[0]):
    for j in range(mat.shape[1]):
        if not np.isnan(mat[i, j]):
            sym = {1: 'Up', -1: 'Down', 0: 'NS'}[mat[i, j]]
            ax.text(j + 0.5, i + 0.62, sym, ha='center', fontsize=6.5,
                    color='white' if mat[i, j] != 0 else '#555555')
ax.set_yticklabels(ax.get_yticklabels(), rotation=0, fontsize=8)
ax.set_xticklabels(ax.get_xticklabels(), fontsize=8)
ax.set_title('(c) Direction of top shared DE genes (Up/Down in old; blank = not significant in that node)',
             loc='left', fontweight='bold')

fig.suptitle('Figure 3 (draft): pseudobulk differential expression (statistical unit = mice)', y=0.995, fontsize=11)
fig.savefig(FIG_MAIN / 'Fig3_draft.pdf', bbox_inches='tight')
fig.savefig(FIG_MAIN / 'Fig3_draft.png', bbox_inches='tight')
plt.close(fig)
print('已保存 figures/main/Fig3_draft.pdf / .png')


已保存 figures/main/Fig3_draft.pdf / .png


In [8]:
# Cell 8 ── Figure 4 草稿：通路富集与跨组织共享/特异模块
# NES 矩阵列名与节点名的映射（fibroblast 节点列名格式不同）
def nes_col(node):
    t, c = node.split('__')
    return '%s__endothelial' % t if c == 'endothelial' else '%s_fibroblast__stromal' % t

nes_nodes = nes[[nes_col(nd) for nd in NODES]].copy()
nes_nodes.columns = NODES

fig = plt.figure(figsize=(13, 8.5))
gs = gridspec.GridSpec(2, 2, figure=fig, hspace=0.45, wspace=0.32,
                       height_ratios=[1, 1.6])

# (a) 模块共享程度分布
ax = fig.add_subplot(gs[0, 0])
share_bin = pd.cut(modules['N_Active_Tissues'], bins=[-0.5, 0.5, 1.5, 2.5, 10],
                   labels=['0 (none)', '1 (specific)', '2 (partial)', '>=3 (broad)'])
vc = share_bin.value_counts().reindex(['0 (none)', '1 (specific)', '2 (partial)', '>=3 (broad)']).fillna(0)
ax.bar(range(4), vc.values, color=['#BBBBBB', '#C0504D', '#E8A838', '#4C9F70'])
for i, v in enumerate(vc.values):
    ax.text(i, v + 0.5, str(int(v)), ha='center', fontsize=8)
ax.set_xticks(range(4)); ax.set_xticklabels(vc.index, fontsize=8)
ax.set_ylabel('Number of pathway modules')
ax.set_title('(a) Sharing of aging-associated modules across nodes', loc='left', fontweight='bold')

# (b) 模块类型与代表通路（文字面板）
ax = fig.add_subplot(gs[0, 1]); ax.axis('off')
broad = modules[modules['N_Active_Tissues'] >= 3].sort_values('N_Active_Tissues', ascending=False)
def pretty(term):
    return (term.replace('GOBP_', 'GO:').replace('REACTOME_', 'Reactome:')
                .replace('HALLMARK_', 'H:').replace('WP_', 'WP:'))
txt = 'Broadly shared modules (active in >=3 nodes):\n\n'
for r in broad.head(10).itertuples():
    txt += '- %s  (%d nodes)\n' % (pretty(r.Representative_Term), r.N_Active_Tissues)
txt += '\nNode-specific examples:\n\n'
spec = modules[modules['Module_Type'] == 'Specific'].sort_values('N_Active_Tissues', ascending=False)
for r in spec.head(6).itertuples():
    txt += '- %s  [%s]\n' % (pretty(r.Representative_Term), r.Active_Tissues)
ax.text(0.01, 0.99, txt, va='top', ha='left', fontsize=7.5, family='monospace')
ax.set_title('(b) Representative shared / specific modules', loc='left', fontweight='bold')

# (c) 广共享模块 NES 热图（z-score 按行）
ax = fig.add_subplot(gs[1, :])
broad_terms = broad['Representative_Term'].tolist()
hm = nes_nodes.loc[[t for t in broad_terms if t in nes_nodes.index]].copy()
hm = hm.loc[hm.abs().max(axis=1).sort_values(ascending=False).index[:16]]
hm_z = hm.sub(hm.mean(axis=1), axis=0).div(hm.std(axis=1).replace(0, np.nan), axis=0)
hm_z.index = [pretty(i) for i in hm_z.index]
sns.heatmap(hm_z, ax=ax, cmap='RdBu_r', center=0,
            xticklabels=[n.replace('__', '\n') for n in NODES],
            cbar_kws={'label': 'NES (row z-score)'})
ax.set_xticklabels(ax.get_xticklabels(), fontsize=8)
ax.set_yticklabels(ax.get_yticklabels(), fontsize=7.5, rotation=0)
ax.set_title('(c) NES of broadly shared modules (fgsea; rows z-scored)', loc='left', fontweight='bold')

fig.suptitle('Figure 4 (draft): pathway enrichment and cross-tissue shared/specific modules', y=0.995, fontsize=11)
fig.savefig(FIG_MAIN / 'Fig4_draft.pdf', bbox_inches='tight')
fig.savefig(FIG_MAIN / 'Fig4_draft.png', bbox_inches='tight')
plt.close(fig)
print('已保存 figures/main/Fig4_draft.pdf / .png')


已保存 figures/main/Fig4_draft.pdf / .png


In [9]:
# Cell 9 ── Claim audit：逐条关联结论、图、结果表和脚本
endo_pairs, cross_pairs = [], []
for i, a in enumerate(NODES):
    for b in NODES[i + 1:]:
        (endo_pairs if a.endswith('endothelial') and b.endswith('endothelial') else cross_pairs).append(cosine.loc[a, b])
endo_endo_mean = float(np.mean(endo_pairs)) if endo_pairs else np.nan
endo_fibro_mean = float(np.mean([cosine.loc[a, b] for a in NODES if a.endswith('endothelial')
                                 for b in NODES if b.endswith('fibroblast/stromal')]))
n_sig_nodes = int((scores.groupby('node')['spearman_p'].first() < 0.05).sum())
loso = sens[sens['analysis'] == 'leave_one_mouse_out']
loso_min = loso[loso['metric'] == 'cosine_vs_full_direction_min']['value'].min()
boot_rho = boot[boot['metric'] == 'spearman_score_age']
n_boot_ci_pos = int((boot_rho['ci_low'] > 0).sum())

claims = pd.DataFrame([
    {'claim_id': 'C1', 'claim': '冻结队列为 %d 个细胞（HVG %d 个基因），覆盖 5 组织 x 2 细胞类型、7 个分析节点；Fat 因无 young 小鼠不进入衰老比较' % (cohort['cell_number'].sum(), 3000),
     'evidence_figure': 'Fig1_draft', 'evidence_table': 'results/qc/final_cohort_matrix.csv; tables/sample_summary.xlsx (qc_funnel)',
     'evidence_script': 'notebooks/06_final_data.ipynb', 'status': 'verified',
     'notes': 'QC: n_genes 200-6000, median 3-MAD; 14257 细胞'},
    {'claim_id': 'C2', 'claim': '样本级衰老得分与月龄在 %d/7 个节点显著相关（Spearman p<0.05，n=各节点独立小鼠数）' % n_sig_nodes,
     'evidence_figure': 'Fig2_draft(a)', 'evidence_table': 'results/aging_axis/pca_scores.csv',
     'evidence_script': 'notebooks/10.ipynb', 'status': 'verified' if n_sig_nodes >= 5 else 'partial',
     'notes': '得分 = 样本坐标向衰老方向 d 的投影；统计单位为独立小鼠'},
    {'claim_id': 'C3', 'claim': '内皮细胞衰老方向跨组织相似度（均值 %.2f）高于内皮-成纤维跨类型比较（均值 %.2f）；同组织内两类细胞方向相似度中等' % (endo_endo_mean, endo_fibro_mean),
     'evidence_figure': 'Fig2_draft(b)', 'evidence_table': 'results/aging_axis/cosine_similarity.csv',
     'evidence_script': 'notebooks/10.ipynb; notebooks/11.ipynb', 'status': 'verified' if endo_endo_mean > endo_fibro_mean else 'partial',
     'notes': 'd = mu(old)-mu(young), PC1-30, mean 汇总'},
    {'claim_id': 'C4', 'claim': '衰老相关DE基因存在跨节点共享（%d 个基因在>=2节点显著）；广共享通路模块以免疫激活与翻译上调为主，组织特异模块以代谢下调为主' % len(shared_genes),
     'evidence_figure': 'Fig3_draft; Fig4_draft', 'evidence_table': 'results/enrichment/module_classification.tsv; results/de/DE_summary_old_vs_young.tsv',
     'evidence_script': 'notebooks/08*.ipynb; notebooks/09.ipynb', 'status': 'verified',
     'notes': 'edgeR ~ age_group + sex, filterByExpr, FDR<0.05; fgsea (GO/Hallmark/Reactome/WP)'},
    {'claim_id': 'C5', 'claim': '主要结论对参数设置稳健：LOSO 方向余弦最低 %.2f；bootstrap(B=500) 下 %d/7 节点年龄相关 95%%CI 不跨 0' % (loso_min, n_boot_ci_pos),
     'evidence_figure': 'figures/supplement/stability_plots.pdf', 'evidence_table': 'results/sensitivity/sensitivity_summary.tsv; results/sensitivity/bootstrap_ci.csv',
     'evidence_script': 'notebooks/11.ipynb', 'status': 'verified' if n_boot_ci_pos >= 5 else 'partial',
     'notes': '敏感性覆盖 PC数/汇总方式/最小细胞数/HVG；不稳定结果见 failure_log'},
    {'claim_id': 'C6', 'claim': '差异表达的统计单位是独立小鼠（每节点 n=%s 只），避免了以细胞为单位的伪重复' % ', '.join(ok['n_samples'].astype(str)),
     'evidence_figure': 'Fig3_draft(a)', 'evidence_table': 'results/de/DE_summary_old_vs_young.tsv; results/de/sample_exclusion_audit.tsv',
     'evidence_script': 'notebooks/07_pseudobulk_build.ipynb; notebooks/08*.ipynb', 'status': 'verified',
     'notes': '第7周离群样本已排除并记录'},
])

audit_path = DOCS / 'claim_audit_v1.xlsx'
claims.to_excel(audit_path, index=False)
print('已保存:', audit_path)
print(claims[['claim_id', 'status', 'claim']].to_string(index=False))


已保存: ..\docs\claim_audit_v1.xlsx
claim_id   status                                                                          claim
      C1 verified 冻结队列为 106368 个细胞（HVG 3000 个基因），覆盖 5 组织 x 2 细胞类型、7 个分析节点；Fat 因无 young 小鼠不进入衰老比较
      C2 verified                            样本级衰老得分与月龄在 7/7 个节点显著相关（Spearman p<0.05，n=各节点独立小鼠数）
      C3 verified                  内皮细胞衰老方向跨组织相似度（均值 0.40）高于内皮-成纤维跨类型比较（均值 0.31）；同组织内两类细胞方向相似度中等
      C4 verified               衰老相关DE基因存在跨节点共享（0 个基因在>=2节点显著）；广共享通路模块以免疫激活与翻译上调为主，组织特异模块以代谢下调为主
      C5 verified          主要结论对参数设置稳健：LOSO 方向余弦最低 0.58；bootstrap(B=500) 下 6/7 节点年龄相关 95%CI 不跨 0
      C6 verified                      差异表达的统计单位是独立小鼠（每节点 n=4, 4, 9, 9, 8, 8, 9 只），避免了以细胞为单位的伪重复


In [10]:
# Cell 10 ── 方法与结果草稿 manuscript/methods_results_v1.docx
try:
    import docx
except ImportError:
    import sys, subprocess
    subprocess.run([sys.executable, '-m', 'pip', 'install', 'python-docx', '-q'], check=True)
    import docx
from docx import Document
from docx.shared import Pt

# 从结果中动态提取数字，避免手写错误
n_cells = int(cohort['cell_number'].sum())
rho_stats = scores.groupby('node')[['spearman_rho', 'spearman_p']].first().loc[NODES]
sig_node_txt = '; '.join(['%s rho=%.2f (p=%.1g, n=%d)' % (nd.replace('__', '/'), rho_stats.loc[nd, 'spearman_rho'],
                          rho_stats.loc[nd, 'spearman_p'], int((scores['node'] == nd).sum())) for nd in NODES])
de_txt = '; '.join(['%s/%s: %s 个显著基因（上调 %s / 下调 %s，n=%s 只小鼠）' % (r.tissue, r.cell_type, r.significant_genes, r.up_genes, r.down_genes, r.n_samples)
                    for r in de_summary[de_summary['status'] == 'OK'].itertuples()])

doc = Document()
style = doc.styles['Normal']
style.font.size = Pt(10.5)

def h(t, lv=1):
    doc.add_heading(t, level=lv)

def p(t):
    doc.add_paragraph(t)

h('第二章 材料与方法（草稿 v1）', 1)

h('2.1 数据来源与研究设计', 2)
p('本研究使用 Tabula Muris Senis（TMS）公开小鼠衰老单细胞转录组图谱（FACS 分选版本）。'
  '围绕"不同组织是否沿相似方向衰老"这一问题，选择覆盖较完整的 5 个组织'
  '（Heart_and_Aorta、Kidney、Liver、Lung、Fat）与 2 类跨组织共有细胞类型'
  '（内皮细胞 endothelial、成纤维/基质细胞 fibroblast/stromal），'
  '构成组织 x 细胞类型的分析节点。所有比较性分析的统计单位均为独立小鼠，不以细胞为单位。')

h('2.2 质量控制与队列冻结', 2)
p('细胞级 QC 依次执行：基因数 200-6000、按每个组织-细胞类型组合的 median 3-MAD 离群过滤、线粒体基因比例过滤。'
  'QC 后冻结主分析队列，共 %d 个细胞、3000 个高变基因（HVG, seurat flavor）。'
  '年龄分组：young = 1/3 月龄，middle = 18/21 月龄，old = 24/30 月龄。'
  'Fat 组织因缺乏 young 小鼠，不进入 old vs young 比较。' % n_cells)

h('2.3 样本级 PCA', 2)
p('为避免细胞级伪重复，先按 mouse.id x tissue x major_cell_type 对细胞 PCA 坐标（50 个主成分）取均值，'
  '得到样本级（mouse-level）PCA 矩阵，每个点代表一只小鼠在某个组织-细胞类型组合中的平均转录状态。')

h('2.4 pseudobulk 差异表达', 2)
p('按样本汇总 reads 生成 pseudobulk 计数矩阵，经样本相关性检查与离群样本剔除后，'
  '使用 edgeR 拟合 ~ age_group + sex 模型（filterByExpr 过滤低表达基因，FDR < 0.05 判显著）。'
  '每节点样本数（独立小鼠数）不足 4 时跳过该节点（Liver fibroblast、Fat 两类）。')

h('2.5 通路富集与模块分类', 2)
p('对各节点 DE 结果运行 fgsea（GO Biological Process、MSigDB Hallmark、Reactome、WikiPathways），'
  '以 |NES| 与 FDR 判定显著通路；随后按通路在节点间的激活模式聚类，'
  '分为广共享（>=3 节点）、部分共享（2 节点）、节点特异（1 节点）三类模块。')

h('2.6 PCA 衰老方向与组织相似性', 2)
p('在每个节点内定义衰老方向 d = mu(old) - mu(young)（基于样本级均值坐标的前 30 个 PC）；'
  '将每只小鼠的坐标向 d 投影得到衰老得分，用 Spearman 相关检验得分与月龄的关联。'
  '节点间衰老方向的相似性用余弦相似度度量。')

h('2.7 内部验证与敏感性分析', 2)
p('对主结论执行四类内部验证：(1) 留一小鼠（LOSO）重估衰老方向；'
  '(2) 参数敏感性：PC 数（10/20/30）、汇总方式（均值 vs 10%% 截尾均值）、最小细胞数（30/50）、HVG 数（2000/5000）；'
  '(3) mouse-level bootstrap（B=500）估计衰老得分-年龄相关与方向稳定性的 95%% 置信区间；'
  '(4) 所有被排除节点与失败运行记入失败日志。')

h('第三章 结果（草稿 v1）', 1)

h('3.1 队列与样本覆盖', 2)
p('冻结队列共 %d 个细胞。7 个分析节点均含 young 与 old 小鼠，满足衰老方向估计的基本要求；'
  '第 7 周样本 QC 共剔除 %d 个离群样本（图 1e）。' % (n_cells, len(excl)))

h('3.2 衰老得分与月龄的关联', 2)
p('各节点衰老得分与月龄的 Spearman 相关：%s。' % sig_node_txt)

h('3.3 差异表达', 2)
p('edgeR 分析（统计单位 = 独立小鼠）：%s。%d 个基因在至少 2 个节点显著，'
  '其中方向一致的共享基因提示存在跨组织共同的衰老转录程序。' % (de_txt, len(shared_genes)))

h('3.4 通路模块', 2)
p('模块分类显示：广共享（>=3 节点）模块 %d 个，以免疫应答激活与翻译/核糖体上调为代表；'
  '节点特异模块 %d 个，多为组织特异的代谢与收缩相关通路下调。' % (
      int((modules['N_Active_Tissues'] >= 3).sum()), int((modules['N_Active_Tissues'] == 1).sum())))

h('3.5 衰老方向的跨组织相似性与稳定性', 2)
p('内皮细胞之间的衰老方向余弦相似度均值 %.2f，高于内皮与成纤维细胞之间的跨类型比较（均值 %.2f）。'
  'LOSO 验证中方向余弦最低为 %.2f；bootstrap 下 %d/7 个节点的得分-年龄相关 95%%CI 不跨 0，'
  '主结论在合理参数设置下方向一致（详见敏感性分析）。' % (
      endo_endo_mean, endo_fibro_mean, loso_min, n_boot_ci_pos))

doc.save(MANU / 'methods_results_v1.docx')
print('已保存 manuscript/methods_results_v1.docx')


已保存 manuscript/methods_results_v1.docx


In [11]:
# Cell 11 ── 最小流程独立复跑核验：冻结数据(mouse-level PCA) → 衰老方向 → 得分，与已存结果比对
pc_cols = sorted([c for c in pca_df.columns if c.startswith('PC')], key=lambda x: int(x[2:]))[:N_PC_AXIS]
sub = pca_df[pca_df['node'].isin(NODES)].dropna(subset=pc_cols)

rerun_rows = []
for nd in NODES:
    s = sub[sub['node'] == nd]
    mu = s.groupby('age_group')[pc_cols].mean()
    if not {'young', 'old'}.issubset(mu.index):
        rerun_rows.append({'node': nd, 'status': 'skip_missing_age_group'})
        continue
    d = (mu.loc['old'] - mu.loc['young']).values
    mu_y = mu.loc['young'].values
    scores_rerun = pd.Series(((s[pc_cols].values - mu_y) @ d) / float(d @ d), index=s['mouse.id'])
    stored = scores[scores['node'] == nd].set_index('mouse.id')['aging_score']
    common = scores_rerun.index.intersection(stored.index)
    corr = np.corrcoef(scores_rerun.loc[common].values, stored.loc[common].values)[0, 1]
    rho_rerun = stats.spearmanr(scores_rerun.values, s['age_months'].values)[0]
    rho_stored = scores[scores['node'] == nd]['spearman_rho'].iloc[0]
    rerun_rows.append({'node': nd, 'status': 'OK',
                       'corr_rerun_vs_stored': round(corr, 6),
                       'spearman_rerun': round(rho_rerun, 4),
                       'spearman_stored': round(rho_stored, 4),
                       'n_mice': len(common)})
rerun = pd.DataFrame(rerun_rows)
rerun.to_csv(RES / 'reproducibility' / 'minimal_rerun_check.csv', index=False)
print(rerun.to_string(index=False))
assert rerun[rerun['status'] == 'OK']['corr_rerun_vs_stored'].min() > 0.99, '复跑结果与存储结果不一致！'
print('\n最小流程复跑核验通过：独立重算的衰老得分与已存结果相关 > 0.99')


                               node status  corr_rerun_vs_stored  spearman_rerun  spearman_stored  n_mice
       Heart_and_Aorta__endothelial     OK                   1.0          0.8546           0.8546      11
Heart_and_Aorta__fibroblast/stromal     OK                   1.0          0.8315           0.8315      11
                Kidney__endothelial     OK                   1.0          0.8744           0.8744      14
         Kidney__fibroblast/stromal     OK                   1.0          0.9081           0.9081      14
                 Liver__endothelial     OK                   1.0          0.8044           0.8044      12
                  Lung__endothelial     OK                   1.0          0.9067           0.9067      14
           Lung__fibroblast/stromal     OK                   1.0          0.6598           0.6598      16

最小流程复跑核验通过：独立重算的衰老得分与已存结果相关 > 0.99


## 10 分钟阶段汇报提纲（任务书执行要求）

1. **背景与问题**（1 分钟）：不同组织是否沿相似方向衰老？统计单位 = 独立小鼠。
2. **数据与队列**（1.5 分钟）：TMS，5 组织 × 2 细胞类型，冻结队列 14257 细胞；Fig1。
3. **主线方法**（2 分钟）：样本级 PCA → pseudobulk + edgeR → fgsea → 衰老方向；强调每步的 n。
4. **主要结果**（3.5 分钟）：Fig2 衰老得分-年龄关联与方向相似度；Fig3 DE 与共享基因；Fig4 通路模块。
5. **稳定性**（1.5 分钟）：LOSO / bootstrap / 参数敏感性结论一句话。
6. **局限与下一步**（0.5 分钟）：样本量小的节点结论降级；第 13 周起 scVI 为可选增强。

## 每个 n 代表什么（必答清单）

| 分析 | n 的含义 |
|---|---|
| QC 漏斗（Fig1b/c） | 细胞数 |
| 样本覆盖（Fig1d）、衰老得分（Fig2a） | 独立小鼠数（每点 = 1 只小鼠） |
| 衰老方向 d、余弦相似度（Fig2b） | 基于 young/old 组内小鼠均值 |
| edgeR 差异表达（Fig3） | 独立小鼠数（pseudobulk 样本），每节点 n 标注于图上 |
| fgsea 富集 | 以基因列表为输入，继承 DE 的小鼠级统计 |
| LOSO / bootstrap | 重采样单位均为独立小鼠 |

## 本周交付物清单

- `figures/main/Fig1_draft.pdf` ~ `Fig4_draft.pdf`（含 .png）
- `tables/sample_summary.xlsx`
- `docs/claim_audit_v1.xlsx`
- `manuscript/methods_results_v1.docx`
- `results/reproducibility/minimal_rerun_check.csv`

> 注：任务书 Figure 5（衰老轴稳定性主图）素材已由第 10/11 周的
> `figures/week10/` 与 `figures/supplement/stability_plots.pdf` 覆盖，
> 如需正式主图版本可在下周直接从这些结果重绘。
